# Extending Python Classes: An Advanced, Step-by-Step Problem Tutorial

**A fresh set of problems with fully worked solutions** · Python 3.10+ · standard library only

This is a teaching notebook, not a collection of long finished scripts. Each problem unfolds the same way a live lesson does: a small starting example, a prediction, one design change, its consequences, a solution, and tests. Run the cells **in order** in a fresh kernel. The code in each numbered problem uses its own named classes so earlier solutions do not get silently replaced by later class definitions.

**Core thread:** extending a class adds behavior and attributes, but inherited code still works on the *actual instance*. That explains dynamic dispatch, class/instance attribute lookup, subclass-aware constructors, abstract contracts, and cooperative `super()`.

## How to work through this notebook

1. Pause at every **Predict** prompt. State the return value or exception before running the next cell.
2. Read each **Your task** step. Try writing the code yourself before comparing it with the cell marked **Solution**.
3. Run the immediate `assert` checks. They are examples of executable specifications, not just output demonstrations.
4. Read the **Why?** cells: understanding method lookup matters more than memorizing the result.
5. Return to the capstone and implement it from the requirements without looking at the subsequent solution cells.

The examples are intentionally distinct from the source lesson's `Person`/`Student` and bank-account examples. The lesson supplies the concepts; the new scenarios test your ability to transfer those concepts. No external packages, files, web requests, or nondeterministic test data are used.

In [1]:
from abc import ABC, abstractmethod
from collections import Counter
from decimal import Decimal

print("Notebook setup: standard-library imports ready")

Notebook setup: standard-library imports ready


## 1 — Extending a base class changes a method the base class *already owns*

**Problem.** A dashboard has a generic card layout. A metric card must add a special data-reading method. The base layout should be reused without copying it into the child class.

**Concept:** `self` in an inherited method is the runtime object, not an instance magically converted into the base class.

### Step 1. Inspect the initial design

The base class knows how to assemble a card, but it has no `reading()` method. This is deliberate: we first observe the limitation before choosing how to express the extension point.

**Predict:** what happens when `BaseCard1("CPU").render()` runs? Would `MetricCard1("CPU", 73).render()` behave differently if only the child provided `reading()`?

In [2]:
class BaseCard1:
    def __init__(self, title: str) -> None:
        self.title = title

    def render(self) -> str:
        return f"[{self.title}] {self.reading()}"


class MetricCard1(BaseCard1):
    def __init__(self, title: str, value: int) -> None:
        super().__init__(title)
        self.value = value

    def reading(self) -> str:
        return f"{self.value}%"

### Step 2. Observe the dispatch

An attribute lookup such as `self.reading` begins with the object and follows Python's lookup rules, including its class and ancestor classes. Because `self` is the `MetricCard1` instance during the inherited `render()` call, the subclass method is found. The plain base instance has no such method and raises `AttributeError`.

In [3]:
metric1 = MetricCard1("CPU", 73)
assert metric1.render() == "[CPU] 73%"
assert isinstance(metric1, BaseCard1)

try:
    BaseCard1("empty").render()
except AttributeError as exc:
    print("Expected missing hook:", type(exc).__name__)
else:
    raise AssertionError("The base instance should have failed")

print(metric1.render())

Expected missing hook: AttributeError
[CPU] 73%


### Step 3. Your task: make the base usable

There are two valid designs, with different contracts: (a) provide a meaningful default `reading()` in the parent; or (b) require subclasses to implement it using `ABC` (Problem 7). Here we need ordinary cards, so choose the **default hook**. Keep the inherited `render()` unchanged.

### Solution, built one method at a time

We do **not** put `if isinstance(self, MetricCard1)` inside the parent. That would make it necessary to edit the parent whenever another card type is added. The extension point is the `reading()` method itself.

In [4]:
class Card1:
    def __init__(self, title: str) -> None:
        self.title = title

    def reading(self) -> str:
        return "no reading"

    def render(self) -> str:
        return f"[{self.title}] {self.reading()}"


class TemperatureCard1(Card1):
    def __init__(self, title: str, celsius: int) -> None:
        super().__init__(title)
        self.celsius = celsius

    def reading(self) -> str:
        return f"{self.celsius} °C"

### Step 4. Verify both paths

Tests should cover the base *and* the extension. A subclass-only test would not detect a broken default. Also check that no override of `render()` was needed.

In [5]:
assert Card1("Overview").render() == "[Overview] no reading"
assert TemperatureCard1("Outside", 18).render() == "[Outside] 18 °C"
assert "render" not in TemperatureCard1.__dict__
print("Problem 1 passed: a new hook customizes inherited behavior")

Problem 1 passed: a new hook customizes inherited behavior


**Why it works.** The base owns the *algorithm* (`render`); each child owns the *variable step* (`reading`). This is a small version of the template-method pattern. It is often simpler than duplicating an entire parent method.

**Challenge:** define a `BatteryCard1` that reports `"91% charged"` without modifying `Card1`. The next cell gives the worked solution.

In [6]:
class BatteryCard1(Card1):
    def __init__(self, title: str, charge: int) -> None:
        super().__init__(title)
        if not 0 <= charge <= 100:
            raise ValueError("charge must be between 0 and 100")
        self.charge = charge

    def reading(self) -> str:
        return f"{self.charge}% charged"


assert BatteryCard1("Laptop", 91).render() == "[Laptop] 91% charged"

## 2 — Extension versus replacement: deciding when to use `super()`

**Problem.** A publishing system logs the result of exporting a document. A secure exporter must add a prefix and preserve the validation already performed by the parent.

Here `super()` means *continue method lookup after this class in the MRO*. In simple single inheritance, that usually invokes the parent's implementation. It is not a synonym for "call my concrete parent by name."

### Step 1. Establish the invariant

The parent rejects an empty document and normalizes line endings. The child must not bypass either guarantee.

**Predict:** what subtle defect would appear if the child implemented `export()` from scratch and returned `self.body` without using `super()`?

In [7]:
class TextExporter2:
    def __init__(self, body: str) -> None:
        self.body = body

    def export(self) -> str:
        if not self.body.strip():
            raise ValueError("document cannot be empty")
        return self.body.replace("\r\n", "\n")


plain2 = TextExporter2("first\r\nsecond")
assert plain2.export() == "first\nsecond"

### Step 2. Your task: extend the result, do not reimplement the algorithm

The secure exporter should return `"SECURE|" + normalized_text`. Name the existing implementation using `super().export()` and transform only its *result*.

### Solution

Keep the entire parent validation in one place. The child adds exactly one responsibility: a marker that external code can inspect.

In [8]:
class SecureExporter2(TextExporter2):
    def export(self) -> str:
        normalized = super().export()
        return "SECURE|" + normalized


secure2 = SecureExporter2("first\r\nsecond")
assert secure2.export() == "SECURE|first\nsecond"

### Step 3. Prove that inherited validation survives

A successful output comparison is insufficient: copying the parent's code could accidentally drop later bug fixes. Exercise the failure case as well.

In [9]:
for cls in (TextExporter2, SecureExporter2):
    try:
        cls("  ").export()
    except ValueError as exc:
        assert str(exc) == "document cannot be empty"
    else:
        raise AssertionError(f"{cls.__name__} bypassed validation")
print("Problem 2 passed: override extends rather than bypasses parent")

Problem 2 passed: override extends rather than bypasses parent


### Step 4. Stretch solution: add another subclass layer

A versioned exporter adds a version stamp *after* the secure marker. This is a small inheritance chain: each level adds behavior, while the initial validation remains centralized.

In [10]:
class VersionedExporter2(SecureExporter2):
    def __init__(self, body: str, version: int) -> None:
        super().__init__(body)
        if version < 1:
            raise ValueError("version must be positive")
        self.version = version

    def export(self) -> str:
        return f"v{self.version}|" + super().export()


assert VersionedExporter2("hello", 4).export() == "v4|SECURE|hello"

**Why it works.** `VersionedExporter2.export()` calls `SecureExporter2.export()`, which continues to `TextExporter2.export()`. The base invariant is applied once. If a child truly needs *replacement*, overriding without `super()` can be appropriate, but that must be a deliberate contract change.

## 3 — An inherited method can use a subclass's class configuration

**Problem.** A renderer uses a per-class line separator and heading symbol. Subclasses change those class constants but reuse the formatting algorithm.

**Prediction target:** compare `Report3.separator` against `self.separator` and `type(self).separator`.

### Step 1. Make configuration part of the class contract

The values are intended to differ *by renderer class*, not by each document instance. We deliberately avoid setting them in `__init__`.

In [11]:
class Report3:
    separator = " / "
    symbol = "#"

    def render(self, title: str, items: list[str]) -> str:
        return f"{self.symbol} {title}: {self.separator.join(items)}"


class CompactReport3(Report3):
    separator = ","
    symbol = ">"


class BulletedReport3(Report3):
    separator = " | "
    symbol = "*"

### Step 2. Predict, then verify

The `render` function is inherited in both children. Does it see the new class attributes? Why would `Report3.separator` hard-coded inside `render` give a different result?

In [12]:
assert Report3().render("Tags", ["a", "b"]) == "# Tags: a / b"
assert CompactReport3().render("Tags", ["a", "b"]) == "> Tags: a,b"
assert BulletedReport3().render("Tags", ["a", "b"]) == "* Tags: a | b"
assert "render" not in CompactReport3.__dict__
print(CompactReport3().render("Tags", ["a", "b"]))

> Tags: a,b


### Step 3. Examine a deliberately broken alternative

When a base method explicitly reads `Report3.separator`, the class name fixes the lookup target. Inherited code can no longer respond to the subclass's value. We keep this mistake in a separate method so the correct implementation remains intact.

In [13]:
def broken_join3(self: Report3, items: list[str]) -> str:
    return Report3.separator.join(items)


assert broken_join3(CompactReport3(), ["a", "b"]) == "a / b"
assert CompactReport3().separator.join(["a", "b"]) == "a,b"

### Step 4. Your task: support a new renderer using *only* class attributes

Create an HTML-oriented renderer whose separator is `"<br>"` and whose symbol is `"H"`. In production you would HTML-escape untrusted input; here the strings are fixed test fixtures so we can focus on inheritance rather than HTML handling.

### Solution and executable check

In a short configuration-only subclass, no overridden method or constructor is required.

In [14]:
class HtmlReport3(Report3):
    separator = "<br>"
    symbol = "H"


assert HtmlReport3().render("Skills", ["Python", "SQL"]) == "H Skills: Python<br>SQL"
print("Problem 3 passed: inherited code uses subclass class attributes")

Problem 3 passed: inherited code uses subclass class attributes


**Why it works.** Ordinary lookup via `self.attribute` checks the instance and, if appropriate, its class/MRO. Later we will deliberately assign an instance attribute with the same name and distinguish that behavior from `type(self).attribute`.

## 4 — Dynamic dispatch can occur two or three calls deep

**Problem.** An alert service builds a notification using a formatter, which asks for a severity tag. A subclass will override just the deepest method.

A frequent mistake is to imagine that an inherited method always invokes other *base-class* methods. The actual receiver matters at every hop.

### Step 1. Trace the call graph on paper

`send → format_message → severity`. Identify which methods are inherited and which method changes in the subclass. Predict the exact string without executing anything.

In [15]:
class Alert4:
    def severity(self) -> str:
        return "INFO"

    def format_message(self, text: str) -> str:
        return f"{self.severity()}: {text}"

    def send(self, text: str) -> str:
        return f"OUTBOX[{self.format_message(text)}]"


class CriticalAlert4(Alert4):
    def severity(self) -> str:
        return "CRITICAL"

### Step 2. Solution: follow the runtime object through every call

`CriticalAlert4().send(...)` finds `send` on `Alert4`, but `self` remains a `CriticalAlert4` instance. Its inherited `format_message` then calls the overridden `severity`.

In [16]:
assert Alert4().send("disk full") == "OUTBOX[INFO: disk full]"
assert CriticalAlert4().send("disk full") == "OUTBOX[CRITICAL: disk full]"
assert "send" not in CriticalAlert4.__dict__
assert "format_message" not in CriticalAlert4.__dict__

### Step 3. Your task: add a child-specific formatter without touching the base

A JSON-like alert will wrap the formatted message in braces. It is not a general JSON serializer: the fixed input avoids escaping concerns. Override `format_message` and **call `super()`** so that the severity from the most-derived class is still used.

### Solution

When an override uses `super().format_message`, the parent formatter still invokes `self.severity()`. That inner call can reach a further override in the most-derived subclass.

In [17]:
class BoxedAlert4(CriticalAlert4):
    def format_message(self, text: str) -> str:
        return "{" + super().format_message(text) + "}"


assert BoxedAlert4().send("disk full") == "OUTBOX[{CRITICAL: disk full}]"
print("Problem 4 passed: virtual dispatch survives nested inherited calls")

Problem 4 passed: virtual dispatch survives nested inherited calls


**Why it works.** Calling `super()` selects the *next implementation* of `format_message`; it does not change the identity of `self`. Separating those two ideas makes many advanced inheritance puzzles straightforward.

## 5 — Instance shadowing versus a subclass-owned policy

**Problem.** An upload service imposes a maximum file size that varies by class. Individual objects are not allowed to relax the policy accidentally by assigning their own `max_bytes` attribute.

The source lesson compares `self.apr` to `type(self).apr`. Here we transfer exactly that lookup distinction into a non-financial application.

### Step 1. A tempting implementation

The first version reads `self.max_bytes`. It honors subclass overrides — but *also* any ordinary instance attribute with the same name. Predict both cases before running the cell.

In [18]:
class Upload5:
    max_bytes = 100

    def accepts(self, size: int) -> bool:
        return 0 <= size <= self.max_bytes


class ImageUpload5(Upload5):
    max_bytes = 500


u5 = ImageUpload5()
assert u5.accepts(400)
u5.max_bytes = 1_000_000
assert u5.accepts(50_000)  # surprising if the limit is a class-level policy

### Step 2. Diagnose the exact mechanism

Assigning `u5.max_bytes` created an *instance* attribute. It did not change `ImageUpload5.max_bytes` or the limit for any other object. Inspect both locations rather than assuming assignment modified the class.

In [19]:
assert u5.__dict__["max_bytes"] == 1_000_000
assert ImageUpload5.max_bytes == 500
assert ImageUpload5().max_bytes == 500
print("instance:", u5.max_bytes, "class:", type(u5).max_bytes)

instance: 1000000 class: 500


### Step 3. Your task: write a method that intentionally uses the runtime class

Read `type(self).max_bytes`. That bypasses the object's ordinary instance dictionary for this lookup, while still selecting a subclass's configured value. It does not make the policy immutable: code can still modify the class attribute unless you add other controls.

### Solution

Decide whether per-object overrides are a feature or a bug before choosing the lookup expression. For this problem, a single limit per class is an explicit business rule.

In [20]:
class PolicyUpload5:
    max_bytes = 100

    def accepts(self, size: int) -> bool:
        if not isinstance(size, int):
            raise TypeError("size must be an integer")
        return 0 <= size <= type(self).max_bytes


class PolicyImageUpload5(PolicyUpload5):
    max_bytes = 500


secure_upload5 = PolicyImageUpload5()
secure_upload5.max_bytes = 1_000_000
assert secure_upload5.accepts(499)
assert not secure_upload5.accepts(501)
assert not PolicyUpload5().accepts(101)

### Step 4. Test failure behavior and explain the boundaries

Negative sizes are invalid inputs here and return `False`. Non-integers are programming errors and produce `TypeError`. A type annotation alone would not enforce that runtime requirement.

In [21]:
assert not secure_upload5.accepts(-1)
try:
    secure_upload5.accepts("500")
except TypeError:
    pass
else:
    raise AssertionError("Noninteger size must fail")
print("Problem 5 passed: class policy is not shadowed by instance assignment")

Problem 5 passed: class policy is not shadowed by instance assignment


**Important nuance.** `type(self).max_bytes` selects the attribute through the class's MRO; it is not a general security sandbox, and descriptor/metaclass behavior can complicate lookups. It is simply the right expression when a policy is intentionally class-scoped.

## 6 — Extend initialization without breaking the parent's invariants

**Problem.** A queue job has a canonical identifier and a state initialized by its base constructor. A scheduled job adds a delay. Some bugs appear only when inherited methods run later, not during construction.

**Design requirement:** all valid subclasses must preserve the original job's initialization rules.

### Step 1. Define the parent invariant

The parent constructor normalizes identifiers and creates an event list. Its `status()` method assumes those attributes exist. This means its constructor is part of the inheritance contract.

In [22]:
class Job6:
    def __init__(self, identifier: str) -> None:
        normalized = identifier.strip().upper()
        if not normalized:
            raise ValueError("identifier required")
        self.identifier = normalized
        self.events: list[str] = ["created"]

    def status(self) -> str:
        return f"{self.identifier}: {len(self.events)} event(s)"


assert Job6(" a-17 ").status() == "A-17: 1 event(s)"

### Step 2. Your task: extend `__init__` safely

Implement `ScheduledJob6` with a nonnegative integer `delay`. Call `super().__init__` exactly once and preserve parent state. Validate the delay before passing it into methods that depend on it.

### Solution, constructor first

Calling the parent initializer is not optional merely because the subclass can assign its *new* attributes. Repeating the parent's assignments by hand duplicates its validation and normalization rules.

In [23]:
class ScheduledJob6(Job6):
    def __init__(self, identifier: str, delay: int) -> None:
        if type(delay) is not int or delay < 0:
            raise ValueError("delay must be a nonnegative integer")
        super().__init__(identifier)
        self.delay = delay
        self.events.append("scheduled")

    def status(self) -> str:
        return f"{super().status()} | delay={self.delay}"

### Step 3. Check both old and new contracts

Test identifier normalization, the original event, the new event, and the extended status. This is more informative than asserting only that the instance can be created.

In [24]:
job6 = ScheduledJob6(" q-5 ", 12)
assert job6.identifier == "Q-5"
assert job6.events == ["created", "scheduled"]
assert job6.status() == "Q-5: 2 event(s) | delay=12"
assert isinstance(job6, Job6)

### Step 4. Test rejected states

Use a small failure matrix: an empty ID is rejected by the *base* contract; a negative delay is rejected by the *extension*. Both errors should be detected before a caller receives a supposedly valid object.

In [25]:
for identifier, delay in [("  ", 1), ("ok", -1), ("ok", True)]:
    try:
        ScheduledJob6(identifier, delay)
    except ValueError:
        pass
    else:
        raise AssertionError((identifier, delay))
print("Problem 6 passed: subclass adds state while preserving base invariants")

Problem 6 passed: subclass adds state while preserving base invariants


**Why it works.** `super().__init__` preserves the inherited state contract; then the child adds `delay` and an event. The extra `type(delay) is int` requirement excludes `bool` (which is a subclass of `int` in Python). This strictness is a choice for this scenario, not a universal rule.

## 7 — An overridable hook called during `__init__` can see a half-built object

**Problem.** A base presenter calls a method to compute its caption during construction. The method is overridden by a subclass that needs fields not yet initialized. This is a common failure in otherwise plausible OOP code.

### Step 1. Read the problematic call order

`Presenter7.__init__` invokes `self.caption()`. But `self` is already the subclass instance, and method dispatch can select the child override **before the child constructor has finished**.

**Predict:** which attribute is missing, and why?

In [26]:
class Presenter7:
    def __init__(self, title: str) -> None:
        self.title = title
        self.cached_caption = self.caption()  # overridable call too early

    def caption(self) -> str:
        return self.title


class NamedPresenter7(Presenter7):
    def __init__(self, title: str, author: str) -> None:
        super().__init__(title)
        self.author = author

    def caption(self) -> str:
        return f"{self.title} — {self.author}"

### Step 2. Reproduce the failure deliberately

Exception demonstrations should catch *only the expected exception*, not silence every error using `except Exception: pass`. Here the missing attribute is `author`.

In [27]:
try:
    NamedPresenter7("Extending classes", "Ada")
except AttributeError as exc:
    assert "author" in str(exc)
    print("Expected early-dispatch failure:", exc)
else:
    raise AssertionError("Expected AttributeError")

Expected early-dispatch failure: 'NamedPresenter7' object has no attribute 'author'


### Step 3. Your task: move the virtual call out of the base constructor

A robust design can compute the caption when needed rather than caching it in a partially initialized object. Have the parent initialize only fields it owns; let the child initialize its fields; then have the inherited public method call `self.caption()` at a safe time.

### Solution: lazy computation through a normal inherited method

Now there is no virtual call inside `__init__`. The child does not need to know about cache invalidation either.

In [28]:
class SafePresenter7:
    def __init__(self, title: str) -> None:
        self.title = title

    def caption(self) -> str:
        return self.title

    def display(self) -> str:
        return f"CARD: {self.caption()}"


class SafeNamedPresenter7(SafePresenter7):
    def __init__(self, title: str, author: str) -> None:
        super().__init__(title)
        self.author = author

    def caption(self) -> str:
        return f"{self.title} — {self.author}"

### Step 4. Confirm the behavior and an additional extension

The public `display()` method stays inherited. A third class changes only the caption while benefiting from the same workflow.

In [29]:
class UpperPresenter7(SafeNamedPresenter7):
    def caption(self) -> str:
        return super().caption().upper()


assert SafePresenter7("Guide").display() == "CARD: Guide"
assert SafeNamedPresenter7("Guide", "Ada").display() == "CARD: Guide — Ada"
assert UpperPresenter7("Guide", "Ada").display() == "CARD: GUIDE — ADA"
print("Problem 7 passed: dispatch happens after initialization")

Problem 7 passed: dispatch happens after initialization


**Why it works.** Python does not delay overridden-method dispatch until constructors finish. Avoid overridable calls in base `__init__` unless the initialization protocol explicitly guarantees all needed state is already available. Lazily computing dependent values is often the simplest repair.

## 8 — Turn an implied extension point into an enforceable abstract contract

**Problem.** Every document processor must define how to parse its input, while all processors share the same `process()` workflow. Constructing a processor that lacks `parse()` should fail immediately, not fail only when the workflow happens to run.

### Step 1. Examine a sentinel that does *not* enforce abstractness

`NotImplemented` is a special singleton value, not an exception. Returning it does not prevent object creation, and it does not automatically require an override. `NotImplementedError` is an exception convention but also does not prohibit instantiation on its own.

In [30]:
class LooseProcessor8:
    def parse(self, text: str):
        return NotImplemented


loose8 = LooseProcessor8()
assert loose8.parse("anything") is NotImplemented
assert isinstance(loose8, LooseProcessor8)

### Step 2. Your task: introduce `ABC` and `@abstractmethod`

Define an abstract `parse(text)` hook and a concrete `process(text)` that calls it and returns a summary. Leave the algorithm in the parent and the data-specific parsing in each child.

### Solution: an abstract hook with a concrete template method

The hook's existence is now checked by Python's abstract-base-class machinery when constructing instances. Type annotations and the ABC contract are separate: the ABC checks implementation presence, not the runtime return type.

In [31]:
class Processor8(ABC):
    @abstractmethod
    def parse(self, text: str) -> list[str]:
        """Return meaningful tokens parsed from input text."""

    def process(self, text: str) -> str:
        tokens = self.parse(text)
        return f"tokens={len(tokens)}; data={','.join(tokens)}"


class CsvProcessor8(Processor8):
    def parse(self, text: str) -> list[str]:
        return [field.strip() for field in text.split(",") if field.strip()]

### Step 3. Verify abstract and concrete construction

A concrete subclass that does not supply `parse` remains abstract. This failure occurs during instantiation rather than later in `process`.

In [32]:
class IncompleteProcessor8(Processor8):
    pass


for abstract_cls in (Processor8, IncompleteProcessor8):
    try:
        abstract_cls()
    except TypeError:
        pass
    else:
        raise AssertionError(f"{abstract_cls.__name__} must be abstract")

assert CsvProcessor8().process(" a, , b ") == "tokens=2; data=a,b"

### Step 4. Your task: add an entirely different parser

A whitespace processor uses `str.split()` and the inherited `process()`. Do not override `process()` just to duplicate its summary logic.

### Solution and tests

One reusable template now supports multiple parsing strategies, with a construction-time guarantee that each concrete subclass provides its parser.

In [33]:
class WhitespaceProcessor8(Processor8):
    def parse(self, text: str) -> list[str]:
        return text.split()


assert WhitespaceProcessor8().process("red   blue\n green") == (
    "tokens=3; data=red,blue,green"
)
assert "process" not in WhitespaceProcessor8.__dict__
print("Problem 8 passed: ABC enforces a subclass hook")

Problem 8 passed: ABC enforces a subclass hook


**Why it works.** Abstract base classes make an inheritance requirement explicit. Choose a default hook (Problem 1) when the parent has meaningful behavior; choose an abstract hook when a concrete child must supply a meaningful implementation. A returned `NotImplemented` value cannot substitute for this contract.

## 9 — An inherited classmethod must construct the requested subclass

**Problem.** A command object can be built from a string such as `"repeat:5"`. We want the factory to stay in the parent while allowing a subclass to inherit it and return an instance of that subclass.

**Key difference:** `cls` in a classmethod refers to the class on which the method was called; `BaseClass(...)` hard-codes a particular class.

### Step 1. Establish compatible constructors

The child inherits the parent's constructor unchanged but overrides the run behavior. This keeps the factory's positional constructor arguments valid for the subclass.

In [34]:
class Command9:
    def __init__(self, action: str, count: int) -> None:
        if not action or count < 0:
            raise ValueError("invalid command")
        self.action = action
        self.count = count

    def run(self) -> str:
        return self.action * self.count


class LoudCommand9(Command9):
    def run(self) -> str:
        return super().run().upper()

### Step 2. Your task: implement the inherited factory

Use `@classmethod`, parse the input in exactly one split, validate its format, and call `cls(action, count)`. In this problem the subclass constructor is compatible with the base constructor. A subclass that adds mandatory arguments would require a different factory contract.

### Solution

A classmethod is inherited like an ordinary method, but its first argument is the class used for the call. The factory has no need to list all known subclasses.

In [35]:
# Final version: define the inherited factory inside the class.
class Command9:
    def __init__(self, action: str, count: int) -> None:
        if not action or count < 0:
            raise ValueError("invalid command")
        self.action = action
        self.count = count

    def run(self) -> str:
        return self.action * self.count

    @classmethod
    def from_spec(cls, spec: str):
        parts = spec.split(":")
        if len(parts) != 2 or not parts[1].isdigit():
            raise ValueError("expected action:nonnegative_count")
        action, raw_count = parts
        return cls(action, int(raw_count))


class LoudCommand9(Command9):
    def run(self) -> str:
        return super().run().upper()

### Step 3. Test both dynamic construction and dynamic dispatch

The earlier prototype is now replaced by the completed class definition, a teaching technique used in the source notebook. In ordinary applications, define the factory inside the class from the start, as shown in the final version above.

In [36]:
base_command9 = Command9.from_spec("go:2")
loud_command9 = LoudCommand9.from_spec("go:2")
assert type(base_command9) is Command9
assert type(loud_command9) is LoudCommand9
assert base_command9.run() == "gogo"
assert loud_command9.run() == "GOGO"

### Step 4. Test bad input and subclass behavior

Parsing errors should produce an explicit error rather than leaking a cryptic unpacking exception. An invalid action such as `":2"` is rejected by the constructor.

In [37]:
for bad in ("nocolon", "go:two", "go:1:extra", ":2"):
    try:
        LoudCommand9.from_spec(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(bad)
print("Problem 9 passed: inherited factory preserves requested subclass")

Problem 9 passed: inherited factory preserves requested subclass


**Why it works.** `cls` is `LoudCommand9` when the factory is invoked as `LoudCommand9.from_spec(...)`. It would instead always return `Command9` if it called `Command9(...)` explicitly. This relies on a documented constructor contract; classmethods cannot automatically solve incompatible subclass signatures.

## 10 — Mutable class attributes: extend configuration without leaking state

**Problem.** A crawler has a list of default file extensions. A subclass adds one extension. Changing the subclass configuration must not rewrite the parent's defaults or contaminate unrelated instances.

This problem separates three layers: the base class default, each subclass default, and each instance's working copy.

### Step 1. Diagnose inherited mutable storage

If a child inherits a list and mutates it in place, both parent and child may still point to the **same list object**. Predict the value of `Crawler10.extensions` after the append.

In [38]:
class Crawler10:
    extensions = [".html", ".txt"]


class MediaCrawler10(Crawler10):
    pass


assert MediaCrawler10.extensions is Crawler10.extensions
MediaCrawler10.extensions.append(".png")
assert Crawler10.extensions == [".html", ".txt", ".png"]

### Step 2. Why simply reassigning the child list changes the situation

`MediaCrawler10.extensions = ...` installs a separate class attribute; `.append(...)` on the inherited list did not. For a new, correct design, make immutable class defaults and copy them into instance-owned working storage.

### Step 3. Your task: separate defaults from mutable working state

Define `allowed_extensions` as a tuple on each class. The base constructor turns the runtime class's tuple into an instance-owned list named `active_extensions`. Add an instance method that modifies only that instance's list.

### Solution

The immutable tuple communicates that the *default policy* is not mutated in place. Copying to a list creates independent mutable state. This is preferable when the workflow genuinely needs per-instance changes.

In [39]:
class SafeCrawler10:
    allowed_extensions = (".html", ".txt")

    def __init__(self) -> None:
        self.active_extensions = list(type(self).allowed_extensions)

    def enable(self, extension: str) -> None:
        if not extension.startswith("."):
            raise ValueError("extension must start with a dot")
        if extension not in self.active_extensions:
            self.active_extensions.append(extension)


class SafeMediaCrawler10(SafeCrawler10):
    allowed_extensions = SafeCrawler10.allowed_extensions + (".png",)

### Step 4. Prove three different forms of isolation

Use two child objects plus one parent object. Modifying one child should change neither the second child nor either class tuple.

In [40]:
left10 = SafeMediaCrawler10()
right10 = SafeMediaCrawler10()
plain10 = SafeCrawler10()
left10.enable(".gif")
left10.enable(".gif")
assert left10.active_extensions == [".html", ".txt", ".png", ".gif"]
assert right10.active_extensions == [".html", ".txt", ".png"]
assert plain10.active_extensions == [".html", ".txt"]
assert SafeCrawler10.allowed_extensions == (".html", ".txt")
assert SafeMediaCrawler10.allowed_extensions == (".html", ".txt", ".png")
print("Problem 10 passed: per-class defaults and per-instance data are isolated")

Problem 10 passed: per-class defaults and per-instance data are isolated


**Why it works.** Inheritance shares access to attributes, not automatic copies of their values. Use immutable defaults for class-scoped configuration and intentionally allocate new mutable objects for per-instance state. Do not assume a subclass declaration implicitly clones an inherited list.

## 11 — Cooperative `super()` in mixins: follow the MRO, not the source file

**Problem.** A notification may be signed and framed. We want two reusable mixins that transform the final output, regardless of how they are combined.

**Advanced concept:** in multiple inheritance, `super()` resumes lookup at the next class in the **method resolution order** (MRO), not necessarily at the lexically named parent.

### Step 1. Define a single terminal implementation

A cooperative chain needs a method that eventually stops calling `super()`. The base `Notice11.render()` is the terminal method here.

In [41]:
class Notice11:
    def __init__(self, text: str) -> None:
        self.text = text

    def render(self) -> str:
        return self.text


assert Notice11("hi").render() == "hi"

### Step 2. Your task: implement two cooperative wrappers

`Signed11` surrounds the next implementation's result with `signed(...)`; `Framed11` surrounds it with `frame[...]`. Each override calls `super().render()` exactly once and uses the resulting string. Do not call `Notice11.render(self)` directly: that would skip an intermediate mixin.

### Solution: each mixin owns only one transformation

The mixins intentionally have no `__init__`; they reuse the final base object's constructor and remain focused on rendering.

In [42]:
class Signed11:
    def render(self) -> str:
        return f"signed({super().render()})"


class Framed11:
    def render(self) -> str:
        return f"frame[{super().render()}]"


class FramedSignedNotice11(Framed11, Signed11, Notice11):
    pass

### Step 3. Predict the method chain

`FramedSignedNotice11` has the lookup order `FramedSignedNotice11 → Framed11 → Signed11 → Notice11 → object`. Thus the *outermost* wrapper is `Framed11`, even though the terminal result originates in `Notice11`.

In [43]:
assert FramedSignedNotice11("hi").render() == "frame[signed(hi)]"
assert [cls.__name__ for cls in FramedSignedNotice11.__mro__] == [
    "FramedSignedNotice11", "Framed11", "Signed11", "Notice11", "object"
]
print(" -> ".join(cls.__name__ for cls in FramedSignedNotice11.__mro__))

FramedSignedNotice11 -> Framed11 -> Signed11 -> Notice11 -> object


### Step 4. Your task: reverse the order without changing either mixin

Create another class that inherits from `Signed11, Framed11, Notice11` in that order. Compare the two strings to observe how MRO determines composition.

### Solution and evidence

Two reusable components yield two different outputs solely from the declared class order. Neither mixin hardcodes the other mixin's name.

In [44]:
class SignedFramedNotice11(Signed11, Framed11, Notice11):
    pass


assert SignedFramedNotice11("hi").render() == "signed(frame[hi])"
assert SignedFramedNotice11("hi").render() != FramedSignedNotice11("hi").render()

### Step 5. Diagnose a non-cooperative override

An override that does not call `super()` can *intentionally* terminate the chain, but that is dangerous for a mixin advertised as composable. We show this isolated counterexample and test the effect.

In [45]:
class ReplaceEverything11:
    def render(self) -> str:
        return "REPLACED"


class BrokenChain11(Framed11, ReplaceEverything11, Signed11, Notice11):
    pass


assert BrokenChain11("hi").render() == "frame[REPLACED]"
print("Problem 11 passed: cooperative calls respect MRO; skipped super breaks chain")

Problem 11 passed: cooperative calls respect MRO; skipped super breaks chain


**Why it works.** Each `super()` uses the entire receiver's MRO. In the broken chain, `ReplaceEverything11.render()` returns without forwarding, so `Signed11` and `Notice11` are skipped. Method order is a public design decision when behavior is composed this way.

## 12 — Initialize a diamond hierarchy exactly once

**Problem.** Two extensions share a common root that records authentication. A combined job needs both extensions, but must not authenticate twice.

The constructors should use a consistent *cooperative* protocol: keyword-only arguments, each class consuming only its own parameters, and `super().__init__(**kwargs)` forwarding the remainder.

### Step 1. Visualize the diamond

`Combined12(Archive12, Audit12)` has two paths to `Root12`. A manual call to `Root12.__init__` from each branch can execute root initialization twice. Python's C3 MRO gives a single linear order if all classes cooperate.

In [46]:
class Root12:
    def __init__(self, *, token: str, **kwargs) -> None:
        if kwargs:
            raise TypeError(f"unexpected arguments: {sorted(kwargs)}")
        if not token:
            raise ValueError("token required")
        self.token = token
        self.trace = ["root"]


class Archive12(Root12):
    def __init__(self, *, archive: str, **kwargs) -> None:
        super().__init__(**kwargs)
        self.archive = archive
        self.trace.append("archive")

### Step 2. Your task: add the second branch cooperatively

`Audit12` consumes `audit_enabled`, forwards the other keywords, and records its own completion. When used in a diamond, its `super()` should continue toward `Root12` exactly once.

### Solution: another single-responsibility initializer

The branch neither names `Root12` explicitly nor assumes that it will be invoked immediately next. That flexibility matters in combined hierarchies.

In [47]:
class Audit12(Root12):
    def __init__(self, *, audit_enabled: bool, **kwargs) -> None:
        super().__init__(**kwargs)
        self.audit_enabled = audit_enabled
        self.trace.append("audit")


class Combined12(Archive12, Audit12):
    def __init__(self, *, name: str, **kwargs) -> None:
        super().__init__(**kwargs)
        self.name = name
        self.trace.append("combined")

### Step 3. Predict the *call order* and the *completion order*

Constructors are entered following MRO, but appends happen *after* the forwarded constructor returns. Therefore trace entries are recorded in the reverse order as the call stack unwinds. Write down both sequences first.

In [48]:
assert [cls.__name__ for cls in Combined12.__mro__] == [
    "Combined12", "Archive12", "Audit12", "Root12", "object"
]
combined12 = Combined12(
    name="daily", archive="cold", audit_enabled=True, token="abc"
)
assert combined12.trace == ["root", "audit", "archive", "combined"]
assert combined12.token == "abc"
assert combined12.archive == "cold"
assert combined12.audit_enabled is True
assert combined12.name == "daily"

### Step 4. Prove root initialization was not duplicated

Here the trace makes the invariant observable. In a real hierarchy, the invariant could instead concern opening a connection, registering a callback, or allocating an expensive resource.

In [49]:
assert combined12.trace.count("root") == 1
assert combined12.trace.count("archive") == 1
assert combined12.trace.count("audit") == 1
print("Problem 12 passed:", " -> ".join(combined12.trace))

Problem 12 passed: root -> audit -> archive -> combined


**Why it works.** `super()` advances through the combined MRO and does not simply call the syntactic parent. Cooperative constructor signatures allow every participating class to consume one piece of the configuration, then forward the rest. The root checks that no arguments were silently discarded.

**Boundary:** this protocol works only when *all participating classes* implement compatible cooperative initialization. It cannot repair an uncooperative third-party base class by itself.

## 13 — A read-only property behaves differently from an ordinary class attribute

**Problem.** A scheduler offers a computed, read-only `capacity` property. A child sets its class-level `base_capacity`. Users must not be able to redefine the computed capacity by writing to an instance.

This exercise goes beyond ordinary class-attribute shadowing by introducing **descriptors**.

### Step 1. Compare ordinary class data with a property

Ordinary class attributes can usually be shadowed by an instance dictionary entry. A property is a descriptor; one without a setter rejects normal assignment, and as a data descriptor it takes precedence over an equally named instance-dictionary entry.

In [50]:
class Scheduler13:
    base_capacity = 4

    @property
    def capacity(self) -> int:
        return type(self).base_capacity * 2


class BatchScheduler13(Scheduler13):
    base_capacity = 7


scheduler13 = BatchScheduler13()
assert scheduler13.capacity == 14

### Step 2. Your task: try three ways to change capacity

Predict the outcome of (a) assigning `scheduler13.base_capacity`, (b) assigning `scheduler13.capacity`, and (c) directly writing `scheduler13.__dict__["capacity"]`. Then distinguish the behavior you *observe* from guarantees about securing an object against malicious code.

### Solution: inspect each lookup separately

`base_capacity` is a normal class attribute. `capacity` is a property defined on the parent, and it is found through the subclass MRO.

In [51]:
# Ordinary class attribute: an instance entry can shadow it.
scheduler13.base_capacity = 1_000
assert scheduler13.base_capacity == 1_000
assert BatchScheduler13.base_capacity == 7
assert scheduler13.capacity == 14  # getter explicitly reads the runtime class

# Read-only property: normal assignment is rejected.
try:
    scheduler13.capacity = 1_000
except AttributeError:
    pass
else:
    raise AssertionError("read-only property assignment should fail")

### Step 3. An advanced lookup experiment

Directly editing `__dict__` is intentionally non-idiomatic. It demonstrates precedence rather than a recommended application API. Because `property` provides descriptor assignment behavior even without a setter, it takes precedence over the same key in the instance dictionary.

In [52]:
scheduler13.__dict__["capacity"] = 999
assert scheduler13.__dict__["capacity"] == 999
assert scheduler13.capacity == 14
assert Scheduler13().capacity == 8
print("Problem 13 passed: the property is resolved before instance-dict entry")

Problem 13 passed: the property is resolved before instance-dict entry


**Why it works.** A property is a *data descriptor*. This is a more specific rule than the simplified "instance attributes always win" shortcut. A property can enforce an ordinary public API, but Python introspection and mutation of class objects mean it is not a security boundary.

## 14 — Validate and register subclass extensions when classes are created

**Problem.** An application supports named encoding strategies. A new encoder should become available to a factory by declaring a subclass, with duplicate keys detected at class-creation time.

`__init_subclass__` is a hook invoked when a subclass is created. We use it for registration, while keeping instance-level encoding behavior polymorphic.

### Step 1. Specify the public extension contract

Every subclass declares a unique keyword-only `key` in the class definition, such as `class UpperCodec14(Codec14, key="upper")`. Every encoder implements `encode(text)`. The factory must reject an unknown name with a useful error.

### Step 2. Your task: implement the base class and registration hook

Create one registry on the base class. In the hook, call `super().__init_subclass__(**kwargs)`, validate the key, reject duplicates, and register `cls`. Do not silently overwrite an existing plugin. The ABC mechanism separately prevents creation of classes that fail to implement `encode`.

### Solution: registration and abstractness solve different problems

`__init_subclass__` is a subclass-creation hook, not a runtime-type checker. It does not automatically verify method signatures or output correctness. We use the hook for keys and the abstract method for implementation presence.

In [53]:
class Codec14(ABC):
    registry: dict[str, type["Codec14"]] = {}

    def __init_subclass__(cls, *, key: str, **kwargs) -> None:
        super().__init_subclass__(**kwargs)
        if not isinstance(key, str) or not key.strip():
            raise ValueError("codec key must be nonempty")
        if key in Codec14.registry:
            raise ValueError(f"duplicate codec key: {key}")
        cls.key = key
        Codec14.registry[key] = cls

    @abstractmethod
    def encode(self, text: str) -> str:
        """Transform the text."""

    @classmethod
    def create(cls, key: str) -> "Codec14":
        try:
            implementation = cls.registry[key]
        except KeyError as exc:
            raise ValueError(f"unknown codec: {key}") from exc
        return implementation()

### Step 3. Add two concrete extensions one at a time

Both implementations share a factory and obey a uniform interface. Neither requires editing the base factory's `if`/`elif` branches because there are no such branches.

In [54]:
class UpperCodec14(Codec14, key="upper"):
    def encode(self, text: str) -> str:
        return text.upper()


class ReverseCodec14(Codec14, key="reverse"):
    def encode(self, text: str) -> str:
        return text[::-1]


assert Codec14.create("upper").encode("Abc") == "ABC"
assert Codec14.create("reverse").encode("Abc") == "cbA"
assert Codec14.registry["upper"] is UpperCodec14

### Step 4. Test duplicate detection and missing implementation

Check a collision in a `try` block. Then declare a subclass with a unique key but no implementation: registration can occur, but ABC instantiation must still fail. Remove that deliberately invalid registry entry afterward to leave a clean demonstration.

In [55]:
try:
    class DuplicateCodec14(Codec14, key="upper"):
        def encode(self, text: str) -> str:
            return text
except ValueError as exc:
    assert "duplicate" in str(exc)
else:
    raise AssertionError("Duplicate key should fail")


class IncompleteCodec14(Codec14, key="incomplete"):
    pass


try:
    Codec14.create("incomplete")
except TypeError:
    pass
else:
    raise AssertionError("An abstract subclass cannot be instantiated")
del Codec14.registry["incomplete"]

### Step 5. Check unknown names and the resulting registry

Validation happens when classes are defined; lookup happens when instances are requested. Keeping those moments separate makes the lifecycle easier to reason about.

In [56]:
try:
    Codec14.create("missing")
except ValueError as exc:
    assert "unknown codec" in str(exc)
else:
    raise AssertionError("unknown key should fail")

assert set(Codec14.registry) == {"upper", "reverse"}
print("Problem 14 passed: subclass registration and ABC checks")

Problem 14 passed: subclass registration and ABC checks


**Why it works.** `__init_subclass__` supports opt-in extension points when a real registry is useful. For small applications where explicit registration is clearer, a plain dictionary may be preferable. The abstract-method mechanism is still needed if you want to prohibit instances lacking a required method.

# 15 — CAPSTONE: Build an extensible assessment-report engine

This is a fresh integration problem that combines the notebook's central ideas without reusing the earlier account or daily-routine examples.

**Scenario.** A learning platform accepts several kinds of assessment. Every completed submission has a learner name, a score, a pass threshold that varies by class, and a rendered report. Quizzes compute percentages; essays supply a moderated score. An honors quiz needs a stricter pass threshold and a special label.

## Capstone requirements, before writing any code

Implement a base class `Submission15` that:

- is abstract: directly constructing it is forbidden;
- strips learner names and rejects blank names;
- requires a `score()` hook returning an integer from 0 through 100;
- supplies a default `label()` returning `"standard"`;
- owns a shared `report()` algorithm returning `"NAME|LABEL|SCORE|PASS"` or `"NAME|LABEL|SCORE|RETRY"`;
- uses `type(self).passing_score`, because passing thresholds belong to classes, not individual instances;
- raises `ValueError` if a subclass returns an out-of-range or non-integer score.

Implement `QuizSubmission15`, `EssaySubmission15`, and `HonorsQuiz15` without copying the shared report algorithm.

### Step 1. Derive the algorithm on paper

For `report()`, the correct sequence is: obtain **one** score → validate the score → read the runtime class's threshold → decide `PASS` or `RETRY` → assemble a string from the object's name, label and score.

Why only one call to `score()`? If a future subclass calculates an expensive or stateful score, multiple evaluations in one report might be inefficient or inconsistent.

### Step 2. Your task: write the base class first

Use `ABC` and `@abstractmethod`. Keep `passing_score = 60` as an ordinary class attribute, and ensure the base constructor initializes the name once. Let the later concrete classes own their specialized inputs.

### Solution, part A — reusable algorithm and explicit contract

This is the most important part of the design: once the parent owns the invariant checks and report layout, adding a new assessment type should require only a constructor and a scoring method.

In [57]:
class Submission15(ABC):
    passing_score = 60

    def __init__(self, learner: str) -> None:
        if not isinstance(learner, str) or not learner.strip():
            raise ValueError("learner name required")
        self.learner = learner.strip()

    @abstractmethod
    def score(self) -> int:
        """Return an integer score from 0 to 100."""

    def label(self) -> str:
        return "standard"

    def report(self) -> str:
        result = self.score()  # exactly one call
        if type(result) is not int or not 0 <= result <= 100:
            raise ValueError("score must be an integer in [0, 100]")
        threshold = type(self).passing_score
        decision = "PASS" if result >= threshold else "RETRY"
        return f"{self.learner}|{self.label()}|{result}|{decision}"

### Step 3. Assert that the base is genuinely abstract

A sentinel result would let callers build an unusable base object. Here instantiation itself is prohibited. This catches incomplete implementations earlier.

In [58]:
try:
    Submission15("Ada")
except TypeError:
    pass
else:
    raise AssertionError("Submission15 must be abstract")

### Step 4. Your task: implement quiz scoring

Quiz data consists of `correct` and `total`. Require integers, `total > 0`, and `0 <= correct <= total`. Return an integer percentage using floor division. In the learner-facing platform you may prefer a different rounding policy; here floor division is specified for deterministic tests.

### Solution, part B — the first concrete extension

The quiz constructor uses `super().__init__(learner)` to inherit normalized learner names. The subclass adds only quiz-specific fields and scoring.

In [59]:
class QuizSubmission15(Submission15):
    def __init__(self, learner: str, correct: int, total: int) -> None:
        if type(correct) is not int or type(total) is not int:
            raise ValueError("correct and total must be integers")
        if total <= 0 or not 0 <= correct <= total:
            raise ValueError("invalid quiz counts")
        super().__init__(learner)
        self.correct = correct
        self.total = total

    def score(self) -> int:
        return (100 * self.correct) // self.total

### Step 5. Test boundaries rather than only the happy path

A complete contract includes 0%, exactly the threshold, and 100%. Also verify a name with extra whitespace gets normalized by the inherited constructor.

In [60]:
assert QuizSubmission15("  Ada  ", 3, 5).report() == "Ada|standard|60|PASS"
assert QuizSubmission15("Bo", 0, 5).report() == "Bo|standard|0|RETRY"
assert QuizSubmission15("Cy", 5, 5).report() == "Cy|standard|100|PASS"
for args in [("Ada", 3, 0), ("Ada", -1, 5), ("Ada", 6, 5), ("Ada", True, 5)]:
    try:
        QuizSubmission15(*args)
    except ValueError:
        pass
    else:
        raise AssertionError(args)

### Step 6. Your task: implement essay scoring without duplicating `report()`

An essay receives a moderated score directly. Validate the input in its constructor and override only `score()`. It must still reuse name validation, label, report layout, and threshold comparison.

### Solution, part C — second concrete extension

The key test below ensures that the essay inherits rather than redefines `report()`. Reusing the parent implementation is a visible part of the exercise, not merely an incidental detail.

In [61]:
class EssaySubmission15(Submission15):
    def __init__(self, learner: str, moderated_score: int) -> None:
        if type(moderated_score) is not int or not 0 <= moderated_score <= 100:
            raise ValueError("invalid essay score")
        super().__init__(learner)
        self.moderated_score = moderated_score

    def score(self) -> int:
        return self.moderated_score


assert EssaySubmission15("Dee", 59).report() == "Dee|standard|59|RETRY"
assert EssaySubmission15("Eli", 60).report() == "Eli|standard|60|PASS"
assert "report" not in EssaySubmission15.__dict__

### Step 7. Your task: specialize the *class policy* and extend a method

An honors quiz passes at 90 rather than 60. It should label itself `"standard-honors"` by extending the inherited `label()` with `super()` instead of replacing its return value from scratch. Do not override `report()` or `score()`.

### Solution, part D — a small specialization with large effects

The inherited report reads `type(self).passing_score`, so honors grading changes without editing the parent or duplicating report logic. The inherited label remains part of the result.

In [62]:
class HonorsQuiz15(QuizSubmission15):
    passing_score = 90

    def label(self) -> str:
        return super().label() + "-honors"


assert HonorsQuiz15("Fox", 17, 20).report() == "Fox|standard-honors|85|RETRY"
assert HonorsQuiz15("Fox", 18, 20).report() == "Fox|standard-honors|90|PASS"
assert "report" not in HonorsQuiz15.__dict__
assert "score" not in HonorsQuiz15.__dict__

### Step 8. Test the shadowing trap

If the honors object's instance dictionary contains `passing_score = 0`, should it suddenly pass an 85% score? Our specification says **no**: the passing score is a class policy. This is why the shared parent uses `type(self).passing_score`.

In [63]:
honors15 = HonorsQuiz15("Gia", 17, 20)
honors15.passing_score = 0
assert honors15.__dict__["passing_score"] == 0
assert type(honors15).passing_score == 90
assert honors15.report() == "Gia|standard-honors|85|RETRY"

### Step 9. Test the parent contract against a defective child

The abstract-method mechanism verifies that an implementation *exists*, but cannot prove that its score is correct. Therefore, the shared report validates the actual returned value. We add a deliberately buggy subclass to test the guard.

In [64]:
class BrokenSubmission15(Submission15):
    def score(self) -> int:
        return 120


try:
    BrokenSubmission15("Hal").report()
except ValueError as exc:
    assert "score must" in str(exc)
else:
    raise AssertionError("The parent must validate subclass results")

### Step 10. Final independent extension exercise

Now extend the system without editing its base: a `FixedScoreSubmission15` stores a valid score and labels itself `"fixed"`. Implement it using the same constructor and hook contract. This is a worked solution, but try writing it first.

### Solution, part E — prove the design is open to extension

A truly extensible base lets you add a new domain variation without editing the shared workflow. This solution needs no `if isinstance(...)` branches in `Submission15.report()`.

In [65]:
class FixedScoreSubmission15(Submission15):
    def __init__(self, learner: str, points: int) -> None:
        if type(points) is not int or not 0 <= points <= 100:
            raise ValueError("invalid points")
        super().__init__(learner)
        self.points = points

    def score(self) -> int:
        return self.points

    def label(self) -> str:
        return "fixed"


assert FixedScoreSubmission15("Ivy", 72).report() == "Ivy|fixed|72|PASS"
assert isinstance(FixedScoreSubmission15("Ivy", 72), Submission15)

### Step 11. Integration test: different types through one interface

Here is the practical payoff of inheritance and polymorphism: client code can iterate over base-typed objects and call `report()` without branching on concrete types. The inherited workflow dispatches to the appropriate score and label implementations.

In [66]:
submissions15: list[Submission15] = [
    QuizSubmission15("Ada", 3, 5),
    EssaySubmission15("Dee", 59),
    HonorsQuiz15("Fox", 18, 20),
    FixedScoreSubmission15("Ivy", 72),
]
reports15 = [submission.report() for submission in submissions15]
assert reports15 == [
    "Ada|standard|60|PASS",
    "Dee|standard|59|RETRY",
    "Fox|standard-honors|90|PASS",
    "Ivy|fixed|72|PASS",
]
print("Problem 15 passed: all assessment types share one workflow")
for report in reports15:
    print("  ", report)

Problem 15 passed: all assessment types share one workflow
   Ada|standard|60|PASS
   Dee|standard|59|RETRY
   Fox|standard-honors|90|PASS
   Ivy|fixed|72|PASS


### Capstone design review: explain each choice

- **Abstract hook:** `score()` is required, because the base cannot produce a real score.
- **Concrete template:** `report()` contains validation, threshold logic, and output structure in one place.
- **Constructor extension:** children call `super().__init__` to keep learner normalization.
- **Subclass-level policy:** `type(self).passing_score` uses the runtime class and resists accidental instance shadowing.
- **Controlled method extension:** the honors label uses `super().label()` to keep parent behavior.
- **Input and output validation:** subclass constructors validate raw inputs; the base additionally validates the `score()` result.
- **Extensibility proof:** the final client loop uses one interface for four different concrete classes.

**Trade-off:** inheritance is appropriate here because the objects genuinely share a public submission contract and a common report workflow. For unrelated behaviors connected only by one shared operation, composition can be simpler.

# Final review — prediction and debugging checkpoints

Attempt these without running the solution cells. They deliberately combine details from several preceding problems. The answers appear immediately afterward so the notebook remains self-contained.

### Checkpoint A — predict five facts

1. When `BoxedAlert4.send()` invokes inherited methods, is `self` an `Alert4` or a `BoxedAlert4` instance?
2. Can `IncompleteProcessor8()` be instantiated merely because it inherits a concrete `process()`?
3. If `Framed11.render()` omits `super()`, will later mixins necessarily execute?
4. If a caller sets `honors15.passing_score = 0`, what threshold does `report()` use?
5. What is the output of `SignedFramedNotice11("hi").render()`?

### Checkpoint A — explained answers

1. `self` is the original **`BoxedAlert4` instance**, regardless of which ancestor defines the executing method.
2. **No.** The abstract `parse()` is not implemented, so construction raises `TypeError`.
3. **No.** A non-cooperative override can stop the chain; later methods are skipped.
4. **90.** `type(self).passing_score` reads the runtime *class* policy, not the instance entry.
5. **`signed(frame[hi])`.** The first mixin in the MRO supplies the outermost wrapper in this cooperative implementation.

### Checkpoint B — a focused executable regression suite

These checks repeat critical contracts across all problems, in one place. Running this cell successfully shows that the tutorial's definitions have not been accidentally overwritten and that the examples remain consistent when executed sequentially.

In [67]:
checks = {
    "hook dispatch": TemperatureCard1("Outside", 18).render() == "[Outside] 18 °C",
    "super validation": SecureExporter2("valid").export() == "SECURE|valid",
    "class configuration": CompactReport3().separator == ",",
    "nested dispatch": BoxedAlert4().send("x") == "OUTBOX[{CRITICAL: x}]",
    "class policy": not secure_upload5.accepts(501),
    "constructor extension": job6.events == ["created", "scheduled"],
    "safe construction": UpperPresenter7("X", "Y").display() == "CARD: X — Y",
    "abstract template": CsvProcessor8().process("a,b") == "tokens=2; data=a,b",
    "subclass-aware factory": type(LoudCommand9.from_spec("x:1")) is LoudCommand9,
    "independent mutable state": ".gif" not in right10.active_extensions,
    "cooperative method chain": FramedSignedNotice11("hi").render() == "frame[signed(hi)]",
    "diamond initialized once": combined12.trace.count("root") == 1,
    "property precedence": scheduler13.capacity == 14,
    "subclass registry": set(Codec14.registry) == {"upper", "reverse"},
    "capstone polymorphism": len(reports15) == 4 and reports15[-1].endswith("PASS"),
}
assert all(checks.values()), {name: passed for name, passed in checks.items() if not passed}
print(f"All {len(checks)} tutorial regression checks passed")

All 15 tutorial regression checks passed


# A reusable design checklist

Before extending a Python class, ask these questions:

1. **Does the parent have a meaningful default?** Add a default hook if yes; use an abstract method when every concrete child must implement it.
2. **Am I adding behavior or replacing it?** Call `super()` when the original behavior and invariants must survive.
3. **Who owns the data?** Use instance attributes for per-object state and explicit class-level configuration for per-class policy.
4. **What should lookups be sensitive to?** `self.setting` may see instance shadowing; `type(self).setting` reads through the runtime class and its MRO; `Base.setting` hard-codes the base.
5. **Is initialization cooperative?** Call the next `__init__`, avoid invoking overridable hooks too early, and agree on a compatible protocol in multiple inheritance.
6. **Do I need a class-level extension mechanism?** An inherited classmethod can construct `cls`; `__init_subclass__` can validate or register subclass declarations when justified.
7. **What invariants must tests establish?** Check ordinary values, edge cases, rejected inputs, construction guarantees, and subclass behavior.

**Next practice:** build a new child of `Submission15` for a timed assessment with a different scoring policy. Add a test for exactly-the-threshold behavior and another for invalid timing input. If you can do so without touching `Submission15.report()`, you have understood the central extension mechanism.